# S0 — 02.04: Polynomial Regression

**Question:** How can engineered polynomial features let ordinary linear regression fit a curve, and what remains linear about that model?

This lesson uses a tiny, synthetic, dimensionless dataset. It demonstrates feature engineering and closed-form OLS; it does not demonstrate performance, generalisation, or extrapolation quality.

## S1 — Continuity from earlier regression lessons

02.01 introduced scalar OLS, 02.02 introduced feature matrices, and 02.03 separated the model, least-squares objective, and optimizer. Here the objective remains least squares. The new step is to create the features `x` and `x²` before fitting.

In [ ]:
import numpy as np

RTOL = 0.0
ATOL = 1e-12
x = np.array([-2.0, -1.0, 0.0, 1.0, 2.0], dtype=np.float64)
y = np.array([4.1, 1.1, 1.6, 2.1, 6.1], dtype=np.float64)


## S2 — Engineer polynomial features

The frozen feature order is `[x, x²]`. For example, `x = -2` becomes `[-2, 4]`. The row correspondence with `y` is preserved.

In [ ]:
X_poly = np.column_stack((x, x**2))
X_poly

## S3 — Build the design matrix

The intercept is included manually in the first column. Thus `X_poly.shape == (5, 2)`, `A.shape == (5, 3)`, and `theta.shape == (3,)`.

In [ ]:
A = np.column_stack((np.ones(x.size, dtype=np.float64), X_poly))
print('X_poly.shape:', X_poly.shape)
print('A.shape:', A.shape)
A

## S4 — Nonlinear in x, linear in parameters

The prediction rule is `y_hat = b + w_1*x + w_2*x²`. It is curved as a function of `x`, but linear in the unknown parameters `[b, w_1, w_2]`: each parameter multiplies a fixed design-matrix column.

## S5 — OLS objective

Use `residuals = y - y_hat`, `RSS = sum(residuals**2)`, and `MSE = RSS / n_samples`. OLS chooses the parameters that minimize this objective.

## S6 — Frozen synthetic example

The degree is 2, the intercept is manual, and no degree search or train/test split is used. These choices keep the arithmetic visible and bounded.

## S7 — Hand calculation of the coefficients

The normal equations are `A.T @ A @ theta = A.T @ y`. For this example:

```text
A.T @ A = [[5, 0, 10], [0, 10, 0], [10, 0, 34]]
A.T @ y = [15, 5, 44]
```

Solving gives `theta = [1.0, 0.5, 1.0]`, in the order `[intercept, x coefficient, x² coefficient]`.

In [ ]:
theta = np.asarray(np.linalg.solve(A.T @ A, A.T @ y), dtype=np.float64)
theta

## S8 — Predictions, residuals, RSS, and MSE

In [ ]:
predictions = np.asarray(A @ theta, dtype=np.float64)
residuals = np.asarray(y - predictions, dtype=np.float64)
rss = np.float64(np.sum(residuals**2, dtype=np.float64))
mse = np.float64(rss / y.size)
print('predictions:', predictions)
print('residuals:', residuals)
print('RSS:', rss)
print('MSE:', mse)

## S9 — Shape-aware NumPy implementation

The private implementation validates NumPy arrays, `float64` dtype, dimensions, non-empty and finite values, matching sample counts, degree 2, compatible shapes, and finite outputs. The notebook keeps the arithmetic visible rather than hiding it behind an estimator.

In [ ]:
def fit_degree_two(x, y):
    if not isinstance(x, np.ndarray) or x.dtype != np.float64 or x.ndim != 1:
        raise TypeError('x must be a one-dimensional np.float64 array')
    if not isinstance(y, np.ndarray) or y.dtype != np.float64 or y.ndim != 1:
        raise TypeError('y must be a one-dimensional np.float64 array')
    if x.size == 0 or y.size == 0 or x.size != y.size:
        raise ValueError('x and y must be non-empty with matching samples')
    if not np.all(np.isfinite(x)) or not np.all(np.isfinite(y)):
        raise ValueError('x and y must be finite')
    features = np.column_stack((x, x**2))
    design = np.column_stack((np.ones(x.size, dtype=np.float64), features))
    coefficients = np.asarray(np.linalg.solve(design.T @ design, design.T @ y), dtype=np.float64)
    fitted = np.asarray(design @ coefficients, dtype=np.float64)
    errors = np.asarray(y - fitted, dtype=np.float64)
    rss_value = np.float64(np.sum(errors**2, dtype=np.float64))
    return coefficients, fitted, errors, rss_value, np.float64(rss_value / y.size)

np.testing.assert_allclose(fit_degree_two(x, y)[0], theta, rtol=RTOL, atol=ATOL)

## S10 — Independent cross-check and sklearn comparison

The private cross-check recomputes transformed features, the design matrix, coefficients, predictions, residuals, losses, and curve values without importing the primary implementation. The library comparison uses `PolynomialFeatures(degree=2, include_bias=False)` followed by `LinearRegression(fit_intercept=True)`. Agreement checks this example; it is not evidence of generalisation.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures

transformer = PolynomialFeatures(degree=2, include_bias=False)
model_features = np.asarray(transformer.fit_transform(x.reshape(-1, 1)), dtype=np.float64)
model = LinearRegression(fit_intercept=True).fit(model_features, y)
np.testing.assert_allclose(model_features, X_poly, rtol=RTOL, atol=ATOL)
np.testing.assert_allclose(model.coef_, theta[1:], rtol=RTOL, atol=ATOL)
np.testing.assert_allclose(model.intercept_, theta[0], rtol=RTOL, atol=ATOL)
print('library coefficients:', model.coef_)
print('library intercept:', model.intercept_)

## S11 — Accessible fitted-curve visual

![Degree-2 polynomial regression on synthetic data](assets/polynomial_fit.png)

**Text equivalent:** observed `y` is `[4.1, 1.1, 1.6, 2.1, 6.1]`; fitted `y_hat` is `[4.0, 1.5, 1.0, 2.5, 6.0]`. Across `x_plot = [-2.0, -1.5, -1.0, -0.5, 0.0, 0.5, 1.0, 1.5, 2.0]`, the fitted values are `[4.0, 2.5, 1.5, 1.0, 1.0, 1.5, 2.5, 4.0, 6.0]`. The visual is a supplementary view of these synthetic, dimensionless calculations.

## S12 — Recap and handoff

Polynomial regression here means ordinary linear regression on engineered features. The curve is nonlinear in `x`, but the parameters enter linearly. The least-squares objective is unchanged. 02.05 will introduce Ridge Regression as a separate regularised objective.